# Forest Species Classification — Multi-Level Model Comparison
Compares **Random Forest** and **MLP (regularized)** across:
- 3 classification **levels** (1 = coarse, 3 = fine-grained species)
- Best feature set selected from Level-3 pilot on raw bands · vegetation-index stats · augmented
- 2 **augmentation strategies**: traditional (geometric) · spectral (band dropout)

**Strategy:** (3 levels and 6 feature sets) × 2 models = 18 runs total:  
- **Phase 1 (6 runs):** Level-3 only × 6 feature sets × 2 models → pick best feature set  
- **Phase 2 (6 runs):** 3 levels × best feature set × 2 models

---
## 1  Imports & Config

In [1]:
import os, warnings
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    f1_score, precision_score, recall_score, average_precision_score
)
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_DIR = "C:\\Users\\uceda\\Documents\\uni\\master\\ENLIGHT\\course\\data\\dataset\\s2\\200m"
TRAIN_LST   = "../data/train_filenames.lst"
TEST_LST    = "../data/test_filenames.lst"
CACHE_DIR   = "../data/cache"
MODEL_DIR   = "../data/models"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Patch geometry ─────────────────────────────────────────────────────────────
MAX_H, MAX_W = 21, 21

# ── Training hyper-params ──────────────────────────────────────────────────────
EPOCHS     = 150
BATCH_SIZE = 32
LR_MIN     = 5e-5
LR_MAX     = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


---
## 2  Taxonomy — species → level mappings

In [2]:
# Level-3 species (20 classes, same order as filenames / original label indices)
species_list = [
    'Abies_alba',           # 0  → silver fir
    'Acer_pseudoplatanus',  # 1  → sycamore maple
    'Alnus_spec',           # 2  → alder
    'Betula_spec',          # 3  → birch
    'Cleared',              # 4  → cleared
    'Fagus_sylvatica',      # 5  → european beech
    'Fraxinus_excelsior',   # 6  → european ash
    'Larix_decidua',        # 7  → european larch
    'Larix_kaempferi',      # 8  → japanese larch
    'Picea_abies',          # 9  → norway spruce
    'Pinus_nigra',          # 10 → black pine
    'Pinus_strobus',        # 11 → weymouth pine
    'Pinus_sylvestris',     # 12 → scots pine
    'Populus_spec',         # 13 → poplar
    'Prunus_spec',          # 14 → cherry
    'Pseudotsuga_menziesii',# 15 → douglas fir
    'Quercus_petraea',      # 16 → sessile oak
    'Quercus_robur',        # 17 → english oak
    'Quercus_rubra',        # 18 → red oak
    'Tilia_spec',           # 19 → linden
]
NUM_CLASSES_L3 = len(species_list)  # 20

# ── Level-2 mapping  (species_idx → level2_label_string) ──────────────────────
# From the hierarchy image:
#   oak            : sessile oak, english oak, red oak           (16,17,18)
#   beech          : european beech                              (5)
#   long-lived dec : sycamore maple, european ash, linden, cherry(1,6,19,14)
#   short-lived dec: alder, poplar, birch                        (2,13,3)
#   spruce/fir     : norway spruce, silver fir                   (9,0)
#   douglas fir    : douglas fir                                 (15)
#   pine           : scots pine, black pine, weymouth pine       (12,10,11)
#   larch          : european larch, japanese larch              (7,8)
#   cleared        : cleared                                     (4)
LEVEL2_MAP = {
    0:  'spruce_fir',
    1:  'long_lived_deciduous',
    2:  'short_lived_deciduous',
    3:  'short_lived_deciduous',
    4:  'cleared',
    5:  'beech',
    6:  'long_lived_deciduous',
    7:  'larch',
    8:  'larch',
    9:  'spruce_fir',
    10: 'pine',
    11: 'pine',
    12: 'pine',
    13: 'short_lived_deciduous',
    14: 'long_lived_deciduous',
    15: 'douglas_fir',
    16: 'oak',
    17: 'oak',
    18: 'oak',
    19: 'long_lived_deciduous',
}

# ── Level-1 mapping  (species_idx → level1_label_string) ──────────────────────
#   broadleaf  : all deciduous + oak + beech                    (1-3,5,6,13,14,16-19)
#   needleleaf : spruce, fir, douglas fir, pine, larch          (0,7-12,15)
#   cleared    : cleared                                        (4)
LEVEL1_MAP = {
    0:  'needleleaf',
    1:  'broadleaf',
    2:  'broadleaf',
    3:  'broadleaf',
    4:  'cleared',
    5:  'broadleaf',
    6:  'broadleaf',
    7:  'needleleaf',
    8:  'needleleaf',
    9:  'needleleaf',
    10: 'needleleaf',
    11: 'needleleaf',
    12: 'needleleaf',
    13: 'broadleaf',
    14: 'broadleaf',
    15: 'needleleaf',
    16: 'broadleaf',
    17: 'broadleaf',
    18: 'broadleaf',
    19: 'broadleaf',
}

def build_level_labels(y_l3: np.ndarray, level_map: dict) -> tuple:
    """Convert integer level-3 labels → integer labels for a coarser level.
    Returns (y_new, class_names) where class_names is sorted alphabetically."""
    unique_names = sorted(set(level_map.values()))
    name2idx     = {n: i for i, n in enumerate(unique_names)}
    y_new = np.array([name2idx[level_map[c]] for c in y_l3], dtype=np.int64)
    return y_new, unique_names

print('Taxonomy loaded.')
print(f'  Level-3 classes : {NUM_CLASSES_L3}')
print(f'  Level-2 classes : {len(set(LEVEL2_MAP.values()))}')
print(f'  Level-1 classes : {len(set(LEVEL1_MAP.values()))}')

Taxonomy loaded.
  Level-3 classes : 20
  Level-2 classes : 9
  Level-1 classes : 3


---
## 3  I/O helpers — loading, caching, vegetation indices

In [3]:
# ── Filename → label index ─────────────────────────────────────────────────────
def filename_to_idx(filename: str) -> int:
    for i, species in enumerate(species_list):
        if species in filename:
            return i
    return 0

# ── Load one .tif as (12, H, W), zero-pad to (12, MAX_H, MAX_W) ──────────────
def load_tif_raw(filename: str) -> np.ndarray:
    """Returns float32 array shaped (12, MAX_H, MAX_W)."""
    path = os.path.join(DATASET_DIR, filename)
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)        # (12, H, W)
    pad_h = MAX_H - img.shape[1]
    pad_w = MAX_W - img.shape[2]
    img = np.pad(img, ((0, 0), (0, pad_h), (0, pad_w)), mode='constant')
    return img                                      # (12, 21, 21)

# ── Vegetation indices ────────────────────────────────────────────────────────
VI_NAMES = ['NDVI', 'EVI', 'NDRE', 'PSRI', 'NDMI', 'SRWI', 'GLI']  # 7 selected

def compute_vi_maps(img: np.ndarray) -> np.ndarray:
    """img: (12, H, W) → returns (7, H, W) for the 7 selected VIs."""
    B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12 = img
    eps = 1e-8
    ndvi = (B8 - B4)  / (B8 + B4  + eps)
    evi  = 2.5 * (B8 - B4) / (B8 + 6*B4 - 7.5*B2 + 1 + eps)
    ndre = (B8 - B7)  / (B8 + B7  + eps)
    psri = (B4 - B3)  / (B8       + eps)
    ndmi = (B8 - B11) / (B8 + B11 + eps)
    srwi = (B11 - B8) / (B11 + B8 + eps)
    gli  = (2*B3 - B2 - B4) / (2*B3 + B2 + B4 + eps)
    return np.stack([ndvi, evi, ndre, psri, ndmi, srwi, gli], axis=0)  # (7,H,W)

def compute_vi_stats(img: np.ndarray) -> np.ndarray:
    """img: (12, H, W) → 7 VIs × 4 stats (mean/std/min/max) = 28 features."""
    vi = compute_vi_maps(img)  # (7, H, W)
    feats = []
    for i in range(7):
        feats += [vi[i].mean(), vi[i].std(), vi[i].min(), vi[i].max()]
    return np.array(feats, dtype=np.float32)  # (28,)

# ── Build / load cached split ─────────────────────────────────────────────────
def build_split(filenames: list, split_name: str):
    """Builds three feature matrices and a shared label vector.
    Caches all to CACHE_DIR as .npy.
    Returns:
        X_raw   (N, 12*21*21=5292)  – flattened bands
        X_idx   (N, 28)             – VI stats
        X_3d    (N, 12, 21, 21)     – 3-D patches (needed for augmentation)
        y       (N,)                – level-3 integer labels
    """
    X_raw_path = os.path.join(CACHE_DIR, f'X_raw_{split_name}.npy')
    X_idx_path = os.path.join(CACHE_DIR, f'X_idx_{split_name}.npy')
    X_3d_path  = os.path.join(CACHE_DIR, f'X_3d_{split_name}.npy')
    y_path     = os.path.join(CACHE_DIR, f'y_{split_name}.npy')

    if all(os.path.exists(p) for p in [X_raw_path, X_idx_path, X_3d_path, y_path]):
        print(f'Loading cached {split_name}...')
        return (
            np.load(X_raw_path),
            np.load(X_idx_path),
            np.load(X_3d_path),
            np.load(y_path),
        )

    print(f'Building {split_name} from .tif files...')
    X_raw_list, X_idx_list, X_3d_list, y_list = [], [], [], []
    for i, fn in enumerate(filenames):
        if i % 1000 == 0:
            print(f'  {i}/{len(filenames)}')
        img = load_tif_raw(fn)               # (12, 21, 21)
        X_raw_list.append(img.flatten())     # (5292,)
        X_idx_list.append(compute_vi_stats(img))  # (28,)
        X_3d_list.append(img)               # (12, 21, 21)
        y_list.append(filename_to_idx(fn))

    X_raw = np.array(X_raw_list, dtype=np.float32)
    X_idx = np.array(X_idx_list, dtype=np.float32)
    X_3d  = np.array(X_3d_list,  dtype=np.float32)
    y     = np.array(y_list,     dtype=np.int64)

    np.save(X_raw_path, X_raw)
    np.save(X_idx_path, X_idx)
    np.save(X_3d_path,  X_3d)
    np.save(y_path,     y)
    print(f'  Saved to {CACHE_DIR}')
    return X_raw, X_idx, X_3d, y

print('I/O helpers defined.')

I/O helpers defined.


---
## 4  Augmentation helpers

In [4]:
# ── Traditional augmentation (geometric) ──────────────────────────────────────
# For nadir remote sensing a canopy patch has no preferred orientation →
# 4 rotations × 2 flips = ×8 factor (original kept, 7 new variants per sample).

def geometric_augment(X_3d: np.ndarray, y: np.ndarray):
    """X_3d: (N, C, H, W).  Returns augmented (N*8, C, H, W) and labels (N*8,)."""
    variants = []
    for k in range(4):                      # 0°, 90°, 180°, 270°
        rot = np.rot90(X_3d, k=k, axes=(2, 3))
        variants.append(rot)
        variants.append(np.flip(rot, axis=3).copy())  # horizontal flip
    X_aug = np.concatenate(variants, axis=0)          # (N*8, C, H, W)
    y_aug = np.tile(y, 8)                             # (N*8,)
    return X_aug.astype(np.float32), y_aug

def flatten_raw(X_3d: np.ndarray) -> np.ndarray:
    """(N, C, H, W) → (N, C*H*W)"""
    N = X_3d.shape[0]
    return X_3d.reshape(N, -1)

def extract_vi_stats_batch(X_3d: np.ndarray) -> np.ndarray:
    """(N, 12, H, W) → (N, 28)"""
    return np.array([compute_vi_stats(x) for x in X_3d], dtype=np.float32)

# ── Spectral augmentation (band dropout) ──────────────────────────────────────
# Randomly zero-out individual bands; forces learning of spectral redundancy.
# Applied on the *flattened* feature vector at dataset construction time so
# the RF and MLP share the same augmented pool.

def spectral_band_dropout(X_3d: np.ndarray, y: np.ndarray,
                           n_copies: int = 7, p_drop: float = 0.2,
                           seed: int = 42) -> tuple:
    """Create n_copies band-dropout copies of each sample.
    Each copy randomly zeros p_drop fraction of the 12 bands.
    Returns concatenation of original + copies: (N*(n_copies+1), C, H, W)."""
    rng = np.random.default_rng(seed)
    N, C, H, W = X_3d.shape
    copies = [X_3d]  # keep originals
    for _ in range(n_copies):
        # for each sample, drop a random subset of bands
        mask = rng.random((N, C, 1, 1)) > p_drop  # (N, C, 1, 1) boolean
        copies.append((X_3d * mask).astype(np.float32))
    X_aug = np.concatenate(copies, axis=0)         # (N*8, C, H, W)
    y_aug = np.tile(y, n_copies + 1)
    return X_aug.astype(np.float32), y_aug

print('Augmentation helpers defined.')

Augmentation helpers defined.


---
## 5  Load data & build all feature sets

In [5]:
with open(TRAIN_LST) as f:
    train_filenames = f.read().splitlines()
with open(TEST_LST) as f:
    test_filenames = f.read().splitlines()
print(f'Train: {len(train_filenames)} | Test: {len(test_filenames)}')

Train: 45337 | Test: 5044


In [6]:
# ── 5.1  Load base arrays (cached after first run) ────────────────────────────
X_raw_train, X_idx_train, X_3d_train, y_l3_train = build_split(train_filenames, 'train')
X_raw_test,  X_idx_test,  X_3d_test,  y_l3_test  = build_split(test_filenames,  'test')

print(f'X_raw_train : {X_raw_train.shape}  X_idx_train : {X_idx_train.shape}')
print(f'X_3d_train  : {X_3d_train.shape}')
print(f'y_l3_train  : {y_l3_train.shape}  (range {y_l3_train.min()}–{y_l3_train.max()})')

Loading cached train...
Loading cached test...
X_raw_train : (45337, 5292)  X_idx_train : (45337, 28)
X_3d_train  : (45337, 12, 21, 21)
y_l3_train  : (45337,)  (range 0–19)


In [ ]:
# ── 5.2  Build augmented 3-D patches ─────────────────────────────────────────
# Traditional (geometric ×8)
X_3d_trad_train, y_trad_train = geometric_augment(X_3d_train, y_l3_train)
print(f'Traditional aug train: {X_3d_trad_train.shape}')

# Spectral (band-dropout ×8, matching geometric factor)
X_3d_spec_train, y_spec_train = spectral_band_dropout(X_3d_train, y_l3_train, n_copies=7)
print(f'Spectral aug train   : {X_3d_spec_train.shape}')

# ── 5.3  Derive flat feature matrices from augmented patches ──────────────────
# Raw
X_raw_trad_train = flatten_raw(X_3d_trad_train)
X_raw_spec_train = flatten_raw(X_3d_spec_train)

# VI stats  (heavier — computed in batch)
print('Computing VI stats for traditional augmented set...')
X_idx_trad_train = extract_vi_stats_batch(X_3d_trad_train)
print('Computing VI stats for spectral augmented set...')
X_idx_spec_train = extract_vi_stats_batch(X_3d_spec_train)

print('All augmented feature sets ready.')
print(f'  X_raw_trad_train: {X_raw_trad_train.shape}  |  X_idx_trad_train: {X_idx_trad_train.shape}')
print(f'  X_raw_spec_train: {X_raw_spec_train.shape}  |  X_idx_spec_train: {X_idx_spec_train.shape}')

Traditional aug train: (362696, 12, 21, 21)


---
## 6  Level label sets

In [ ]:
# ── Level-3 (fine-grained species, 20 classes) ─────────────────────────────────
LEVELS = {
    3: {
        'train': y_l3_train,
        'test' : y_l3_test,
        'classes': species_list,
    },
}

# Level-2 and Level-1
for lvl, lmap in [(2, LEVEL2_MAP), (1, LEVEL1_MAP)]:
    y_tr, cls = build_level_labels(y_l3_train, lmap)
    y_te, _   = build_level_labels(y_l3_test,  lmap)
    LEVELS[lvl] = {'train': y_tr, 'test': y_te, 'classes': cls}

# Also build level labels for the augmented sets
for lvl, lmap in [(3, None), (2, LEVEL2_MAP), (1, LEVEL1_MAP)]:
    if lvl == 3:
        LEVELS[3]['trad_train'] = y_trad_train
        LEVELS[3]['spec_train'] = y_spec_train
    else:
        y_trad, _ = build_level_labels(y_trad_train, lmap)
        y_spec, _ = build_level_labels(y_spec_train, lmap)
        LEVELS[lvl]['trad_train'] = y_trad
        LEVELS[lvl]['spec_train'] = y_spec

for lvl in [1, 2, 3]:
    nc = len(LEVELS[lvl]['classes'])
    nt = len(LEVELS[lvl]['train'])
    nte = len(LEVELS[lvl]['test'])
    print(f'Level {lvl}: {nc} classes | train={nt} | test={nte}')

---
## 7  Model definitions & training utilities

In [ ]:
# ── MLP (regularized) ─────────────────────────────────────────────────────────
class ImprovedMLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden_dim: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

# ── Metrics helper ─────────────────────────────────────────────────────────────
def compute_metrics(y_true: np.ndarray, y_pred_prob: np.ndarray) -> dict:
    """y_true: (N,) int labels; y_pred_prob: (N, C) probs."""
    y_pred = np.argmax(y_pred_prob, axis=1)
    nc     = y_pred_prob.shape[1]
    y_oh   = np.eye(nc)[y_true]
    m = {}
    for avg in ('micro', 'weighted'):
        m[f'precision_{avg}'] = precision_score(y_true, y_pred, average=avg, zero_division=0)
        m[f'recall_{avg}']    = recall_score(   y_true, y_pred, average=avg, zero_division=0)
        m[f'f1_{avg}']        = f1_score(       y_true, y_pred, average=avg, zero_division=0)
        m[f'mAP_{avg}']       = average_precision_score(y_oh, y_pred_prob, average=avg)
    return m

# ── Class-weighted loss builder ────────────────────────────────────────────────
def class_weights_from_labels(y: np.ndarray, nc: int) -> torch.Tensor:
    """Inverse-frequency weights, normalised so mean weight = 1."""
    counts = np.bincount(y, minlength=nc).astype(np.float32)
    counts = np.where(counts == 0, 1, counts)   # avoid /0
    w = 1.0 / counts
    w = w / w.mean()                             # normalise
    return torch.tensor(w, dtype=torch.float32).to(device)

# ── MLP training ───────────────────────────────────────────────────────────────
def train_mlp(X_tr: np.ndarray, y_tr: np.ndarray,
               X_te: np.ndarray, y_te: np.ndarray,
               nc: int, tag: str = '') -> tuple:
    """Normalise, train ImprovedMLP, return (results_dict, history_dict)."""
    # normalise
    mean = X_tr.mean(axis=0)
    std  = X_tr.std(axis=0) + 1e-8
    Xtr_n = ((X_tr - mean) / std).astype(np.float32)
    Xte_n = ((X_te - mean) / std).astype(np.float32)

    # tensors
    Xtr_t = torch.tensor(Xtr_n)
    ytr_t = torch.tensor(y_tr, dtype=torch.long)
    Xte_t = torch.tensor(Xte_n)
    yte_t = torch.tensor(y_te, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=BATCH_SIZE, shuffle=False)

    # model
    model = ImprovedMLP(in_dim=X_tr.shape[1], out_dim=nc).to(device)

    # class-weighted CE loss to handle imbalance
    weights   = class_weights_from_labels(y_tr, nc)
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR_MIN)
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer, base_lr=LR_MIN, max_lr=LR_MAX,
        step_size_up=len(train_loader) * 4,
        mode='triangular', cycle_momentum=False,
    )

    history = {'train_loss': [], 'test_loss': [],
               'f1_micro': [], 'f1_weighted': [],
               'mAP_micro': [], 'mAP_weighted': []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss += loss.item() * len(Xb)
        train_loss /= len(train_loader.dataset)

        model.eval()
        test_loss = 0.0
        probs_all, labels_all = [], []
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                logits = model(Xb)
                test_loss += criterion(logits, yb).item() * len(Xb)
                probs_all.append(torch.softmax(logits, dim=1).cpu().numpy())
                labels_all.append(yb.cpu().numpy())
        test_loss /= len(test_loader.dataset)

        y_prob = np.concatenate(probs_all,  axis=0)
        y_true = np.concatenate(labels_all, axis=0)
        m = compute_metrics(y_true, y_prob)

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['f1_micro'].append(m['f1_micro'])
        history['f1_weighted'].append(m['f1_weighted'])
        history['mAP_micro'].append(m['mAP_micro'])
        history['mAP_weighted'].append(m['mAP_weighted'])

        if epoch % 25 == 0 or epoch == 1:
            print(f'  [{tag}] Ep {epoch:3d}/{EPOCHS} | '
                  f'TrLoss {train_loss:.4f} TeLoss {test_loss:.4f} | '
                  f'F1w {m["f1_weighted"]:.4f} mAPw {m["mAP_weighted"]:.4f}')

    # final eval (best softmax)
    results = {}
    for avg in ('micro', 'weighted'):
        results[avg] = {
            'precision': precision_score(y_true, np.argmax(y_prob, 1), average=avg, zero_division=0),
            'recall':    recall_score(   y_true, np.argmax(y_prob, 1), average=avg, zero_division=0),
            'f1':        f1_score(       y_true, np.argmax(y_prob, 1), average=avg, zero_division=0),
            'mAP':       average_precision_score(np.eye(nc)[y_true], y_prob, average=avg),
        }
    return results, history


# ── RF training ────────────────────────────────────────────────────────────────
def train_rf(X_tr: np.ndarray, y_tr: np.ndarray,
              X_te: np.ndarray, y_te: np.ndarray,
              nc: int, tag: str = '') -> dict:
    """Train RF (class_weight='balanced'), return results dict."""
    print(f'  [{tag}] Training Random Forest...')
    rf = RandomForestClassifier(
        n_estimators=100, max_depth=20, class_weight='balanced',
        random_state=42, n_jobs=-1,
    )
    rf.fit(X_tr, y_tr)
    y_pred  = rf.predict(X_te)
    y_proba = rf.predict_proba(X_te)   # (N, nc)
    y_oh    = np.eye(nc)[y_te]

    results = {}
    for avg in ('micro', 'weighted'):
        results[avg] = {
            'precision': precision_score(y_te, y_pred, average=avg, zero_division=0),
            'recall':    recall_score(   y_te, y_pred, average=avg, zero_division=0),
            'f1':        f1_score(       y_te, y_pred, average=avg, zero_division=0),
            'mAP':       average_precision_score(y_oh, y_proba, average=avg),
        }
    acc = accuracy_score(y_te, y_pred)
    print(f'  [{tag}] Accuracy {acc:.4f} | F1w {results["weighted"]["f1"]:.4f} '
          f'mAPw {results["weighted"]["mAP"]:.4f}')
    return results

print('Model definitions ready.')

---
## 8  Main experiment loop
**Phase 1:** Level-3 × 6 feature sets × 2 models → select best feature set (highest weighted F1, averaged over RF+MLP).  
**Phase 2:** 3 levels × best feature set × 2 models.  
Models are saved to `MODEL_DIR` after each run.

In [ ]:
import joblib

# ── Feature set registry ───────────────────────────────────────────────────────
FEATURE_SETS = {
    'raw':            (X_raw_train,      X_raw_test,  'train',      'test'),
    'indices':        (X_idx_train,      X_idx_test,  'train',      'test'),
    'trad_augmented': (X_raw_trad_train, X_raw_test,  'trad_train', 'test'),
    'spec_augmented': (X_raw_spec_train, X_raw_test,  'spec_train', 'test'),
    'trad_aug_idx':   (X_idx_trad_train, X_idx_test,  'trad_train', 'test'),
    'spec_aug_idx':   (X_idx_spec_train, X_idx_test,  'spec_train', 'test'),
}

# all_results[(level, feat_set, model)] = results_dict
all_results   = {}
all_histories = {}   # MLP only

# ── Helper: save RF and MLP models ────────────────────────────────────────────
def save_rf(model, tag: str):
    path = os.path.join(MODEL_DIR, f'rf_{tag}.joblib')
    joblib.dump(model, path)
    print(f'  Saved RF  → {path}')

def save_mlp(model, tag: str):
    path = os.path.join(MODEL_DIR, f'mlp_{tag}.pt')
    torch.save(model.state_dict(), path)
    print(f'  Saved MLP → {path}')

# ── Wrap train_rf / train_mlp to also return the model object ─────────────────
def train_rf_and_save(Xtr, y_tr, Xte, y_te, nc, tag):
    print(f'  [{tag}] Training Random Forest...')
    rf = RandomForestClassifier(
        n_estimators=100, max_depth=20, class_weight='balanced',
        random_state=42, n_jobs=-1,
    )
    rf.fit(Xtr, y_tr)
    y_pred  = rf.predict(Xte)
    y_proba = rf.predict_proba(Xte)
    y_oh    = np.eye(nc)[y_te]
    results = {}
    for avg in ('micro', 'weighted'):
        results[avg] = {
            'precision': precision_score(y_te, y_pred, average=avg, zero_division=0),
            'recall':    recall_score(   y_te, y_pred, average=avg, zero_division=0),
            'f1':        f1_score(       y_te, y_pred, average=avg, zero_division=0),
            'mAP':       average_precision_score(y_oh, y_proba, average=avg),
        }
    acc = accuracy_score(y_te, y_pred)
    print(f'  [{tag}] Accuracy {acc:.4f} | F1w {results["weighted"]["f1"]:.4f} '
          f'mAPw {results["weighted"]["mAP"]:.4f}')
    save_rf(rf, tag.replace('/', '_'))
    return results

def train_mlp_and_save(Xtr, y_tr, Xte, y_te, nc, tag):
    results, history, model = train_mlp(Xtr, y_tr, Xte, y_te, nc, tag=tag, return_model=True)
    save_mlp(model, tag.replace('/', '_'))
    return results, history

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1: Level-3 × all 6 feature sets × 2 models  →  pick best feature set
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*70)
print('PHASE 1 — Level-3 pilot: finding best feature set')
print('='*70)

lvl = 3
nc  = len(LEVELS[lvl]['classes'])
y_te = LEVELS[lvl]['test']

for fs_name, (Xtr, Xte, y_tr_key, y_te_key) in FEATURE_SETS.items():
    y_tr = LEVELS[lvl][y_tr_key]
    tag  = f'L{lvl}/{fs_name}'
    print(f'\n--- {tag}  (train={len(y_tr)}, feats={Xtr.shape[1]}) ---')

    rf_res = train_rf_and_save(Xtr, y_tr, Xte, y_te, nc, tag=tag+'/RF')
    all_results[(lvl, fs_name, 'RF')] = rf_res

    mlp_res, mlp_hist = train_mlp_and_save(Xtr, y_tr, Xte, y_te, nc, tag=tag+'/MLP')
    all_results[(lvl, fs_name, 'MLP')] = mlp_res
    all_histories[(lvl, fs_name)]      = mlp_hist

# ── Select best feature set (avg weighted F1 over RF + MLP at Level 3) ────────
fs_scores = {}
for fs_name in FEATURE_SETS:
    f1_rf  = all_results.get((3, fs_name, 'RF'),  {}).get('weighted', {}).get('f1', 0)
    f1_mlp = all_results.get((3, fs_name, 'MLP'), {}).get('weighted', {}).get('f1', 0)
    fs_scores[fs_name] = (f1_rf + f1_mlp) / 2

best_fs = max(fs_scores, key=fs_scores.get)
print(f'\nFeature-set scores (avg RF+MLP weighted F1 @ Level 3):')
for fs, sc in sorted(fs_scores.items(), key=lambda x: -x[1]):
    marker = '  ← BEST' if fs == best_fs else ''
    print(f'  {fs:20s}: {sc:.4f}{marker}')
print(f'\n→ Best feature set: {best_fs}')

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2: All 3 levels × best feature set × 2 models
#          (Level-3 results already in all_results — skip re-training)
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*70)
print(f'PHASE 2 — All levels with best feature set: {best_fs}')
print('='*70)

Xtr_best, Xte_best, y_tr_key_best, y_te_key_best = FEATURE_SETS[best_fs]

for lvl in [1, 2]:   # Level-3 already done in Phase 1
    nc   = len(LEVELS[lvl]['classes'])
    y_te = LEVELS[lvl][y_te_key_best]
    y_tr = LEVELS[lvl][y_tr_key_best]
    tag  = f'L{lvl}/{best_fs}'
    print(f'\n{"="*70}')
    print(f'LEVEL {lvl}  —  {nc} classes: {LEVELS[lvl]["classes"]}')
    print(f'--- {tag}  (train={len(y_tr)}, feats={Xtr_best.shape[1]}) ---')

    rf_res = train_rf_and_save(Xtr_best, y_tr, Xte_best, y_te, nc, tag=tag+'/RF')
    all_results[(lvl, best_fs, 'RF')] = rf_res

    mlp_res, mlp_hist = train_mlp_and_save(Xtr_best, y_tr, Xte_best, y_te, nc, tag=tag+'/MLP')
    all_results[(lvl, best_fs, 'MLP')] = mlp_res
    all_histories[(lvl, best_fs)]      = mlp_hist

print('\n✓ All experiments finished.')
print(f'   Phase 1: 6 feature sets × Level 3 × 2 models = 12 runs')
print(f'   Phase 2: 2 remaining levels × best feature set ({best_fs}) × 2 models = 4 runs')
print(f'   Total  : 16 runs  (vs 36 in the original full-grid approach)')

---
## 9  Results summary table

In [ ]:
rows = []
for (lvl, fs, mdl), res in sorted(all_results.items()):
    w = res['weighted']
    rows.append({
        'Level'  : lvl,
        'Features': fs,
        'Model'  : mdl,
        'F1_w'   : round(w['f1'],        4),
        'mAP_w'  : round(w['mAP'],       4),
        'Prec_w' : round(w['precision'], 4),
        'Rec_w'  : round(w['recall'],    4),
    })

df_results = pd.DataFrame(rows)
df_results = df_results.sort_values(['Level', 'Features', 'Model'])

print('\nFull results (weighted metrics):')
with pd.option_context('display.max_rows', 100, 'display.float_format', '{:.4f}'.format):
    display(df_results)

csv_path = os.path.join(MODEL_DIR, 'all_results.csv')
df_results.to_csv(csv_path, index=False)
print(f'Saved to {csv_path}')

---
## 10  Visualisations

In [ ]:
# ── 10.1  Grouped bar: F1 (weighted) across all combinations ─────────────────
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

fs_order    = ['raw', 'indices',
               'trad_augmented', 'spec_augmented',
               'trad_aug_idx',   'spec_aug_idx']
fs_labels   = ['Raw', 'VI\nStats',
               'Trad\n(raw)', 'Spec\n(raw)',
               'Trad\n(idx)', 'Spec\n(idx)']
lvl_colors  = {1: '#4C72B0', 2: '#55A868', 3: '#C44E52'}
mdl_hatches = {'RF': '', 'MLP': '////'}

x     = np.arange(len(fs_order))
width = 0.13

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_axisbelow(True)
ax.yaxis.grid(True, alpha=0.3, linestyle='--')

offsets = {}
slot    = 0
for lvl in [1, 2, 3]:
    for mdl in ['RF', 'MLP']:
        vals = [all_results.get((lvl, fs, mdl), {}).get('weighted', {}).get('f1', 0)
                for fs in fs_order]
        offset = x + (slot - 2.5) * width
        ax.bar(offset, vals, width, label=f'L{lvl} {mdl}',
               color=lvl_colors[lvl], hatch=mdl_hatches[mdl],
               alpha=0.85, zorder=3)
        slot += 1

ax.set_xticks(x)
ax.set_xticklabels(fs_labels, fontsize=10)
ax.set_ylabel('Weighted F1', fontsize=11)
ax.set_title('Weighted F1 — All Levels × Feature Sets × Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, ncol=6, framealpha=0.9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'comparison_f1_grouped.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10.2  Heatmap grid: Level × Feature set, one subplot per model ────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, mdl in zip(axes, ['RF', 'MLP']):
    mat = np.zeros((3, len(fs_order)))
    for i, lvl in enumerate([1, 2, 3]):
        for j, fs in enumerate(fs_order):
            mat[i, j] = all_results.get((lvl, fs, mdl), {}).get('weighted', {}).get('f1', 0)

    im = ax.imshow(mat, vmin=0, vmax=1, cmap='YlGn', aspect='auto')
    ax.set_xticks(range(len(fs_order)))
    ax.set_xticklabels(fs_labels, fontsize=9)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['Level 1', 'Level 2', 'Level 3'], fontsize=10)
    ax.set_title(f'{mdl} — Weighted F1', fontsize=11, fontweight='bold')
    for i in range(3):
        for j in range(len(fs_order)):
            ax.text(j, i, f'{mat[i, j]:.3f}', ha='center', va='center',
                    fontsize=8, color='black')

fig.colorbar(im, ax=axes, fraction=0.03, pad=0.04, label='Weighted F1')
plt.suptitle('Weighted F1 Heatmap — RF vs MLP', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'comparison_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10.3  MLP training curves for raw data across all levels ──────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True)
epochs_x  = range(1, EPOCHS + 1)

for row, lvl in enumerate([1, 2, 3]):
    hist = all_histories.get((lvl, 'raw'), {})
    if not hist:
        continue
    axes[row, 0].plot(epochs_x, hist['train_loss'], label='Train')
    axes[row, 0].plot(epochs_x, hist['test_loss'],  label='Test')
    axes[row, 0].set_title(f'L{lvl} Loss')
    axes[row, 0].legend(); axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].plot(epochs_x, hist['f1_micro'],    label='micro')
    axes[row, 1].plot(epochs_x, hist['f1_weighted'], label='weighted')
    axes[row, 1].set_title(f'L{lvl} F1')
    axes[row, 1].set_ylim(0, 1); axes[row, 1].legend(); axes[row, 1].grid(True, alpha=0.3)

    axes[row, 2].plot(epochs_x, hist['mAP_micro'],    label='micro')
    axes[row, 2].plot(epochs_x, hist['mAP_weighted'], label='weighted')
    axes[row, 2].set_title(f'L{lvl} mAP')
    axes[row, 2].set_ylim(0, 1); axes[row, 2].legend(); axes[row, 2].grid(True, alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Epoch')

plt.suptitle('MLP Training Curves (raw features) — per Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'mlp_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10.4  Augmentation effect: raw vs trad_aug vs spec_aug for each level+model
aug_sets    = ['raw', 'trad_augmented', 'spec_augmented']
aug_labels  = ['Raw', 'Traditional\nAug', 'Spectral\nAug']
colors      = ['#4C72B0', '#55A868', '#C44E52']

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)

for col, lvl in enumerate([1, 2, 3]):
    for row, mdl in enumerate(['RF', 'MLP']):
        ax  = axes[row, col]
        f1v = [all_results.get((lvl, fs, mdl), {}).get('weighted', {}).get('f1',  0) for fs in aug_sets]
        mpv = [all_results.get((lvl, fs, mdl), {}).get('weighted', {}).get('mAP', 0) for fs in aug_sets]
        x   = np.arange(3)
        w   = 0.35
        ax.set_axisbelow(True)
        ax.yaxis.grid(True, alpha=0.3, linestyle='--')
        b1 = ax.bar(x - w/2, f1v,  w, label='F1 (w)',  color=colors, alpha=0.9,  zorder=3)
        b2 = ax.bar(x + w/2, mpv,  w, label='mAP (w)', color=colors, alpha=0.45, zorder=3)
        for bar, v in [(b, val) for bs, vals in [(b1, f1v), (b2, mpv)] for b, val in zip(bs, vals)]:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(aug_labels, fontsize=8)
        ax.set_title(f'{mdl}  Level {lvl}', fontsize=10, fontweight='bold')
        ax.set_ylim(0, 1.0)
        ax.spines[['top', 'right']].set_visible(False)
        if col == 0:
            ax.set_ylabel('Score', fontsize=9)

from matplotlib.patches import Patch
legend_els = [Patch(facecolor='grey', alpha=0.9, label='Solid = F1'),
              Patch(facecolor='grey', alpha=0.45, label='Light = mAP')]
fig.legend(handles=legend_els, fontsize=9, loc='upper right', framealpha=0.9)
plt.suptitle('Augmentation Effect: Raw vs Traditional vs Spectral\n(weighted F1 & mAP)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'augmentation_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10.5  Level effect: coarse (L1) → fine (L3) for raw data ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x  = np.arange(3)
w  = 0.35

for ax, mdl in zip(axes, ['RF', 'MLP']):
    f1v  = [all_results.get((lvl, 'raw', mdl), {}).get('weighted', {}).get('f1',  0) for lvl in [1,2,3]]
    mapv = [all_results.get((lvl, 'raw', mdl), {}).get('weighted', {}).get('mAP', 0) for lvl in [1,2,3]]
    cols = ['#4C72B0', '#55A868', '#C44E52']
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, alpha=0.3, linestyle='--')
    b1 = ax.bar(x - w/2, f1v,  w, color=cols, alpha=0.9,  zorder=3, label='F1')
    b2 = ax.bar(x + w/2, mapv, w, color=cols, alpha=0.45, zorder=3, label='mAP')
    for bars, vals in [(b1, f1v), (b2, mapv)]:
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Level 1\n(3 cls)', 'Level 2\n(9 cls)', 'Level 3\n(20 cls)'], fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_title(f'{mdl} — Coarse → Fine (raw features)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Score (weighted)', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Classification Difficulty by Level\n(weighted F1 & mAP, raw features)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'level_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()